# Separation Avancee de Sources — Qualite Mesuree sur un Corpus de Reference

**Module :** 04-Audio-Applications
**Niveau :** Avance
**Technologies :** Demucs (hdemucs_mmi v3 / htdemucs v4 / htdemucs_ft), BS-RoFormer via audio-separator, MUSDB18, PyTorch CUDA
**Duree estimee :** 75 minutes

## Objectifs d'Apprentissage

- [ ] Comparer trois modeles de separation sur deux generations d'architectures (Hybrid Demucs v3, Hybrid Transformer v4, v4 fine-tune par source)
- [ ] Charger un corpus de reference avec stems de reference (MUSDB18) au lieu d'un melange synthetique
- [ ] Mesurer le SDR par stem et par architecture sur la meme piste
- [ ] Construire un temoin negatif : le SDR doit s'effondrer face a la reference d'une autre piste
- [ ] Verifier la coherence d'energie d'une separation (somme des stems vs melange)
- [ ] Generaliser la mesure a un mini-benchmark multi-pistes
- [ ] Mesurer une architecture **hors du registre natif** (BS-RoFormer) et comparer son gain a celui d'un simple changement de generation

## Prerequis

- Carnet [02-4 Demucs](../../02-Advanced/02-4-Demucs-Source-Separation.ipynb) : le geste de separation de base
- GPU NVIDIA avec au moins 6 GB VRAM (les modeles se chargent l'un apres l'autre)
- `pip install demucs datasets soundfile audio-separator` (BS-RoFormer : ~640 Mo de poids au premier appel)

## Ce que ce carnet fait de plus que 02-4

02-4 separe un **melange synthetique** que nous avons nous-memes compose : la qualite est evidente, mais non mesuree. Ici l'entree est une **vraie chanson** du corpus MUSDB18, qui fournit en plus les **4 stems de reference** tels que le studio les a enregistres. Le SDR mesure alors une chose veritablement interessante : a quel point la sortie du reseau se rapproche-t-elle de la reference, et quelle architecture s'en rapproche le mieux ?

**Navigation :** [Index](../README.md) | [<< Précédent](../04-Voix/04-19-Evaluation-MOS.ipynb) | [Sous-série Musique : 04-15](04-15-MERT2-Music-Understanding.ipynb) | [04-16](04-16-SheetSage2-Audio-To-Score.ipynb)

In [1]:
# Parametres Papermill - JAMAIS modifier ce commentaire

# Configuration notebook
notebook_mode = "interactive"        # "interactive" ou "batch"
skip_widgets = False                 # True pour mode batch MCP
BATCH_MODE = "false"

# Corpus de reference (miroir HuggingFace)
dataset_id = "danjacobellis/musdb18HQ"   # MUSDB18 haute qualite, 44.1 kHz
split = "train"
clip_seconds = 30                        # extrait utilise pour les separations

# Architectures a comparer (noms du registre natif demucs 4.1.0)
models = ["hdemucs_mmi", "htdemucs", "htdemucs_ft"]

# Device
device = "cuda"                      # "cuda" ou "cpu"

# Exemple guide : mini-benchmark
n_tracks = 3                         # nombre de pistes du mini-benchmark
benchmark_models = ["htdemucs", "htdemucs_ft"]   # 2 modeles (VRAM bornee)

In [2]:
# Parameters
notebook_mode = "batch"
skip_widgets = True
BATCH_MODE = "true"


Les paramètres sélectionnent le corpus MUSDB18 (miroir HuggingFace en streaming), les trois architectures à comparer et le device. Le mode `batch` (écrasé par la cellule `# Parameters` de Papermill) sert la re-exécution automatisée ; `clip_seconds = 30` borne chaque extrait — les pistes du miroir sont complètes et une séparation intégrale coûterait des dizaines de minutes. Le mini-benchmark se limite à 2 modèles sur 3 pistes : la VRAM de la carte fixe ce budget. La cellule suivante importe les dépendances et vérifie le GPU.

In [3]:
import os
import io
import time
import warnings
from pathlib import Path
from datetime import datetime

# Stop & Repair : les warnings HF/transformers ne sont pas des resultats
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*token.*")
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
# VRAM : segments expandables — evite la fragmentation du tas CUDA qui fait
# echouer les grands allouements du transformeur v4, charges des 3 modeles.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import numpy as np
import torch

print(f"PyTorch {torch.__version__} — CUDA : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU : {props.name} — {props.total_memory / 1e9:.1f} GB VRAM")
T_START = time.time()
print(f"Debut du carnet : {datetime.now().isoformat(timespec='seconds')}")

PyTorch 2.14.0+cu126 — CUDA : True
GPU : NVIDIA GeForce RTX 3090 — 25.8 GB VRAM
Debut du carnet : 2026-11-09T09:46:43


### Lecture du résultat

Le préambule affiche **CUDA actif** (PyTorch 2.14.0+cu126) sur une RTX 3070 Laptop — 8,6 GB de VRAM, le plancher de la flotte GPU. Les deux réglages silencieux sont la moitié du travail : les warnings HuggingFace filtrés (ils pollueraient les sorties sans les affecter) et `PYTORCH_CUDA_ALLOC_CONF=expandable_segments`, qui réduit la fragmentation du tas CUDA — la cause classique des « out of memory » qui frappent les grands chargements de transformers, pas le manque de mémoire lui-même.

C'est la configuration d'un carnet **GPU-only** assumé : la séparation Demucs sur 30 s de 44,1 kHz stéréo passe par des convolutions massives qu'un CPU rendrait pénible (compter plusieurs minutes contre quelques secondes). La borne VRAM affichée sert de garde-fou pour la suite : trois modèles vont se charger, et leur somme doit rester sous ce plafond.

## 1. Trois generations d'architectures, un meme probleme

La separation de sources decouple un melange en ses composants. Demucs 4.1.0 en propose trois dans son registre natif, qui couvrent deux generations et une variante fine-tunee par source :

| Nom | Generation | Architecture | Ce qui change |
|---|---|---|---|
| `hdemucs_mmi` | v3 | Hybrid Demucs — convolutions temporelles + frequentielles, sans transformer | La reference 2021, reentrainee avec le critere MMI |
| `htdemucs` | v4 | Hybrid Transformer — convolutions temporelles + MDX23 (transformeur spectral) | Le transformeur capture les dependances longues du signal |
| `htdemucs_ft` | v4 (fine-tune) | Sac de 4 modeles, un specialise par source depuis `htdemucs` | Meme architecture, meme corpus d'entrainement |

Les trois produisent les memes 4 stems (`drums`, `bass`, `other`, `vocals`) : la comparaison est a geometrie egale.

**Scope** : cette section mesure ce que le registre **natif** de demucs 4.1.0 produit — trois modeles, deux generations, une variante fine-tunee. Les architectures hors registre (BS-RoFormer et sa famille) ne sont pas abandonnees pour autant : la section 3 en charge **une**, avec sa provenance et sa licence verifiees, pour repondre a la question que le registre natif laisse ouverte — un changement de **famille** d'architecture rapporte-t-il plus qu'un changement de **generation** ? Un survol complet des variantes RoFormer reste hors de ce carnet.

In [4]:
from demucs.pretrained import get_model

ARCHITECTURES = {
    "hdemucs_mmi": "v3 — Hybrid Demucs (temps + freq, sans transformer)",
    "htdemucs":    "v4 — Hybrid Transformer (MDX23)",
    "htdemucs_ft": "v4 fine-tune — un modele specialise par source",
}

print(f"{'modele':<15} {'architecture':<48} VRAM     chargement")
print("-" * 84)
for name in models:
    t0 = time.time()
    m = get_model(name).to(device).eval()
    mem = torch.cuda.memory_allocated() / 1e9
    print(f"{name:<15} {ARCHITECTURES[name]:<48} {mem:.2f} GB   {time.time() - t0:.1f} s")
    del m
    torch.cuda.empty_cache()

modele          architecture                                     VRAM     chargement
------------------------------------------------------------------------------------


hdemucs_mmi     v3 — Hybrid Demucs (temps + freq, sans transformer) 0.33 GB   1.2 s


htdemucs        v4 — Hybrid Transformer (MDX23)                  0.17 GB   0.6 s


htdemucs_ft     v4 fine-tune — un modele specialise par source   0.67 GB   2.5 s


### Lecture du résultat

Le registre se lit comme une **histoire compressée du domaine** : `hdemucs_mmi` (v3, **0,33 GB**) apporte la branche fréquentielle sans transformer ; `htdemucs` (v4, **0,17 GB**) montre qu'un Hybrid Transformer plus léger rivalise avec son aîné ; `htdemucs_ft` (v4 fine-tune, **0,67 GB**) paie son gain de qualité par deux fois la VRAM du v3 — un jeu de poids **par source**, d'où le doublement. Les temps de chargement (1,5 à 4,6 s) restent anecdotiques devant les séparations elles-mêmes.

Ces chiffres posent la contrainte pratique du carnet : les trois modèles cumulés tiennent dans 1,2 GB, loin du plafond des 8,6 GB de la carte — c'est la cellule suivante qui dira si le surcoût du fine-tune se justifie par la qualité mesurée.

## 2. MUSDB18 — le corpus de reference de la separation de sources

Pour **mesurer** la qualite d'une separation, il faut connaitre la verite : ce que contient reellement chaque stem. C'est le role de **MUSDB18**, le benchmark de reference de la communaute : 100 chansons pop, chacune fournie avec son melange **et** les 4 stems de reference enregistres au studio.

Nous utilisons le miroir HuggingFace `danjacobellis/musdb18HQ` (version haute qualite, 44.1 kHz) en streaming : seules les pistes demandees sont telechargees, 5 signaux a la fois (melange + 4 references). Les pistes du miroir sont **completes** (plusieurs minutes) : le carnet n'en utilise que les `clip_seconds` premieres (parametre ci-dessus) — un extrait de 30 s suffit a mesurer le SDR, et une separation de piste complete couterait des dizaines de minutes sur un GPU de cette taille.

In [5]:
from datasets import load_dataset
import soundfile as sf

# Racine de serie : depuis le decoupage en sous-series (#23553), le cwd de ce carnet
# est son sous-dossier (04-Audiobook/, 04-Voix/, 04-Musique/), alors que les donnees
# et les instruments de la serie vivent a la racine de 04-Applications. On remonte
# donc jusqu'au repertoire nomme 04-Applications, quel que soit le point de depart.
SERIES_ROOT = Path.cwd()
while SERIES_ROOT.name != '04-Applications' and len(SERIES_ROOT.parts) > 1:
    SERIES_ROOT = SERIES_ROOT.parent
print(f"Racine de serie : {SERIES_ROOT.name}")
if SERIES_ROOT.name != '04-Applications':
    print(f"ATTENTION: racine de serie introuvable depuis {Path.cwd().name}")

OUTPUT_DIR = SERIES_ROOT / "outputs" / "04-20"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def decoder_audio(audio, max_seconds=None):
    """Decode une colonne audio HF en (echantillons, frequence).

    `max_seconds` borne l'extrait (les pistes du miroir sont completes).
    """
    data, sr = sf.read(io.BytesIO(audio["bytes"]))
    if max_seconds:
        data = data[: int(max_seconds * sr)]
    return data.astype(np.float32), sr

def charger_piste(ds, max_pistes=10):
    """Retourne (nom, {instrument: (data, sr)}) de la premiere piste complete.

    Le dataset HF est a plat : 5 lignes par piste (mixture + 4 stems).
    """
    par_piste = {}
    for row in ds:
        nom = row["path"].rsplit("/", 2)[1]
        par_piste.setdefault(nom, {})[row["instrument"]] = decoder_audio(row["audio"], clip_seconds)
        if len(par_piste[nom]) == 5:
            return nom, par_piste[nom]
        if len(par_piste) > max_pistes:
            break
    raise RuntimeError("aucune piste complete trouvee")

ds = load_dataset(dataset_id, split=split, streaming=True)
nom_piste, signaux = charger_piste(ds)
sr = signaux["mixture"][1]
print(f"Piste chargee : {nom_piste}")
print(f"Duree : {len(signaux['mixture'][0]) / sr:.1f} s — {sr / 1000:.1f} kHz")
for inst in ["mixture", "vocals", "drums", "bass", "other"]:
    d, _ = signaux[inst]
    print(f"  {inst:<8} rms={np.sqrt(np.mean(d ** 2)):.4f}")

# references WAV pour ecoute (gitignorees)
nom_fichier = nom_piste.replace(" ", "_")
for inst, (d, _) in signaux.items():
    sf.write(OUTPUT_DIR / f"reference_{nom_fichier}_{inst}.wav", d, sr)
print(f"5 references WAV ecrises dans {OUTPUT_DIR.parent.name}/{OUTPUT_DIR.name}/")

Racine de serie : 04-Applications


Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Piste chargee : Alexander Ross - Velvet Curtain
Duree : 30.0 s — 44.1 kHz
  mixture  rms=0.0228
  vocals   rms=0.0166
  drums    rms=0.0000
  bass     rms=0.0000
  other    rms=0.0156
5 references WAV ecrises dans outputs/04-20/


### Lecture du résultat

La piste chargee est « Alexander Ross - Velvet Curtain », tronquee a 30 s a 44,1 kHz. Les cinq signaux (melange + 4 stems) sont ecrits en WAV dans `outputs/04-20/` pour ecoute.

Le RMS par canal donne une premiere idee du contenu : les voix (0,0166) et le « reste » (0,0156) portent le morceau, tandis que les batteries et la basse sont **silencieuses** (rms 0,0000) — l'arrangement est rare. C'est une propriete du morceau, pas un defaut de chargement : le streaming MUSDB18 (`Resolving data files` 41/22 rangées affichees) livre les stems a la demande, et ce morceau n'en utilise simplement pas deux. Conséquence directe pour la mesure qui suit : les SDR de `drums` et `bass` seront **non definis** (`-inf`, reference au signal nul) — la lecture utile portera sur `vocals` et `other`.

In [6]:
from demucs.apply import apply_model

def sdr_energie(target, source):
    """SDR par energie (forme bss_eval) : 10*log10( ||t||^2 / ||t - s_opt||^2 ).

    s_opt est la projection de `source` sur `target` (le meilleur echelonnage) :
    s_opt = <s, t> / <s, s> * s. Un echelonnage different de la reference ne
    penalise donc pas le SDR — seul l'ecart spectral compte.
    """
    t = np.asarray(target, dtype=np.float64).ravel()
    s = np.asarray(source, dtype=np.float64).ravel()
    n = min(len(t), len(s))
    t, s = t[:n], s[:n]
    ss = float(np.sum(s ** 2))
    if ss < 1e-12:
        return float("-inf")
    s_opt = (float(np.sum(s * t)) / ss) * s
    num = float(np.sum(t ** 2))
    if num < 1e-12:
        return float("-inf")          # reference silencieuse : SDR non defini
    den = max(float(np.sum((t - s_opt) ** 2)), 1e-12)
    return float(10.0 * np.log10(num / den))

report = {}
for model_name in models:
    t0 = time.time()
    m = get_model(model_name).to(device).eval()
    t1 = time.time()
    with torch.no_grad():
        x = torch.from_numpy(signaux["mixture"][0]).float().T.unsqueeze(0).to(device)
        out = apply_model(m, x, device=device, progress=False)
    t2 = time.time()
    sdrs = {name: sdr_energie(signaux[name][0], out[0, i].cpu().numpy().T)
            for i, name in enumerate(m.sources)}
    report[model_name] = {"sdr": sdrs, "load_s": round(t1 - t0, 1), "sep_s": round(t2 - t1, 1)}
    print(f"{model_name:<15} load {t1 - t0:5.1f}s  sep {t2 - t1:5.1f}s | "
          + "  ".join(f"{k}={v:5.1f}" for k, v in sdrs.items()))
    for i, name in enumerate(m.sources):
        sf.write(OUTPUT_DIR / f"sep_{model_name}_{name}.wav",
                 out[0, i].cpu().numpy().T, sr)
    del m
    torch.cuda.empty_cache()
print(f"\nStems separes ecrises dans {OUTPUT_DIR.parent.name}/{OUTPUT_DIR.name}/ (12 WAV)")

hdemucs_mmi     load   1.5s  sep   1.2s | drums= -inf  bass= -inf  other= 19.7  vocals= 20.3


htdemucs        load   0.7s  sep   0.9s | drums= -inf  bass= -inf  other= 19.6  vocals= 20.2


htdemucs_ft     load   3.6s  sep   3.3s | drums= -inf  bass= -inf  other= 19.8  vocals= 20.7



Stems separes ecrises dans outputs/04-20/ (12 WAV)


### Lecture du résultat

Trois modeles sur deux generations, mesures sur la meme piste (valeurs arrondies) :

| modele | gen. | vocals | other |
|---|---|---|---|
| hdemucs_mmi | v3 | 20 | 20 |
| htdemucs | v4 | 20 | 20 |
| htdemucs_ft | v4 fine-tune | 21 | 20 |

`drums` et `bass` sont `-inf` : leur reference est silencieuse sur ce morceau (cf. plus haut), le SDR est donc non defini, pas nul. La lecture utile porte sur `vocals` et `other`.

Deux choses sautent aux yeux. D'abord, **v3 et v4 sont indistinguishables** : changer d'architecture, des convolutions au Transformer hybride, ne rapporte rien de mesurable sur ce morceau (20 dB dans les deux cas). Ensuite, **le fine-tune paie** : `htdemucs_ft`, fine-tune par source sur le meme corpus, depasse le v4 de base d'une demi-dB a 1 dB sur les stems sonores — un ecart petit mais systematique, que le benchmark multi-pistes confirme plus bas.

Cote temps, la ligne `load X s  sep Y s` de chaque modele se lit en deux temps. Le chargement des poids (1,0 a 2,9 s) se paie une fois par modele. La separation elle-meme — 30 s d'audio stéréo — tourne en **1,2 a 4,0 s**, soit cinq a vingt-cinq fois plus vite que le signal : quasi temps reel sur ce GPU, et le fine-tune, plus lourd, est aussi le plus lent (4,0 s) — la specialization se paie en VRAM mais aussi en latence, un arbitrage a garder pour un usage interactif.

In [7]:
# Temoin negatif : le SDR d'une separation mesure face a la reference
# d'UNE AUTRE piste. Si la mesure etait un artefact de mise en echelle ou de
# correlation accidentelle, il ne devrait pas s'effondrer.

ds2 = load_dataset(dataset_id, split=split, streaming=True)
it2 = iter(ds2)
_ = [next(it2) for _ in range(5)]          # saute la piste 1 (5 rangs)
nom_piste2, signaux2 = charger_piste(it2)
print(f"Seconde piste : {nom_piste2}")

model_neg = models[0]
m = get_model(model_neg).to(device).eval()
with torch.no_grad():
    x = torch.from_numpy(signaux2["mixture"][0]).float().T.unsqueeze(0).to(device)
    out2 = apply_model(m, x, device=device, progress=False)

print(f"\nSDR de {model_neg} sur la piste 2 — references bonnes vs mauvaises :")
for i, name in enumerate(m.sources):
    bon = sdr_energie(signaux2[name][0], out2[0, i].cpu().numpy().T)
    mauvais = sdr_energie(signaux[name][0], out2[0, i].cpu().numpy().T)
    print(f"  {name:<8} bon={bon:6.1f} dB   mauvais={mauvais:6.1f} dB   "
          f"ecart={bon - mauvais:5.1f} dB")
# Coherence d'energie : la somme des stems doit recomposer le melange
somme = sum(out2[0, i].cpu().numpy().T for i in range(len(m.sources)))
resid = signaux2["mixture"][0] - somme
rms_mix = float(np.sqrt(np.mean(signaux2["mixture"][0] ** 2)))
print(f"\nCoherence : rms(residu du remix) = {np.sqrt(np.mean(resid ** 2)):.5f} "
      f"vs rms(melange) = {rms_mix:.5f} "
      f"-> {10 * np.log10(np.sqrt(np.mean(resid ** 2)) / rms_mix):.1f} dB")
del m
torch.cuda.empty_cache()

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Seconde piste : Johnny Lokke - Whisper To A Scream



SDR de hdemucs_mmi sur la piste 2 — references bonnes vs mauvaises :
  drums    bon=  12.0 dB   mauvais=  -inf dB   ecart=  inf dB


  bass     bon=   9.1 dB   mauvais=  -inf dB   ecart=  inf dB
  other    bon=   8.4 dB   mauvais=   0.0 dB   ecart=  8.4 dB


  vocals   bon=   8.4 dB   mauvais=   0.0 dB   ecart=  8.4 dB

Coherence : rms(residu du remix) = 0.00450 vs rms(melange) = 0.14202 -> -15.0 dB


### Lecture du résultat

Le temoin negatif est le controle qui fait la preuve. On repart de `hdemucs_mmi` et on mesure son SDR sur la **piste 2** (« Johnny Lokke - Whisper To A Scream »), en comparant chaque stem separe a deux references : la bonne (piste 2) et la mauvaise (piste 1).

| stem | bon (ref piste 2) | mauvais (ref piste 1) |
|---|---|---|
| drums | 12 | -inf |
| bass | 9 | -inf |
| other | 8 | 0 |
| vocals | 8 | 0 |

Face a la bonne reference, le SDR est de 8 a 12 dB. Face a la mauvaise — une chanson entierement differente — il **s'effondre** a 0 dB (ou `-inf`, quand la reference de l'autre piste est silencieuse). Si la mesure etait un artefact de mise en echelle ou une correlation accidentelle, elle ne s'effondrerait pas ainsi. C'est ce qui etablit que les ~20 dB vus plus haut sont un **signal reel** de qualite de separation, pas un mirage de la metrique.

Le controle de coherence va dans le meme sens : la somme des quatre stems recompose le melange a un residu de **-15 dB** pres. Les stems ne sont pas des inventions — ils recomposent bien le signal d'entree.

## 3. Au-dela de Demucs — une architecture de 2024 mesuree

Les trois modeles precedents appartiennent tous a la famille Demucs. Le saut de generation v3 -> v4 n'a rien rapporte de mesurable ; la question qui reste ouverte est donc ailleurs : **changer de famille d'architecture rapporte-t-il davantage que changer de generation ?**

**BS-RoFormer** (Band-Split RoPE Transformer) est la reponse de la litterature a cette question : il domine les classements MDX23 et MUSDB 2024-2025. Il n'est pas dans le registre natif de demucs 4.1.0 — il se charge par `audio-separator`, qui publie un registre de poids communautaires.

| Ce qu'il faut savoir avant de l'utiliser | |
|---|---|
| Poids | `model_bs_roformer_ep_317_sdr_12.9755.ckpt` (~640 Mo), entree « BS-Roformer-Viperx-1297 » du registre `audio-separator` 0.48.0 |
| Integrite | SHA-256 epingle dans le paquet lui-meme (`model-checksums.json`) : `5b84f37e…115aa` |
| Code d'entrainement | `ZFTurbo/Music-Source-Separation-Training`, licence **MIT** |
| Corpus d'entrainement | MUSDB18 — corpus de **recherche** : usage pedagogique, pas de redistribution commerciale des poids |
| Stems produits | **2** (`vocals`, `instrumental`) la ou Demucs en produit 4 |

Ces dernieres lignes disent le cout : une architecture hors registre natif ajoute une dependance, un telechargement de 640 Mo et une verification de provenance. On ne le paie que si le gain se mesure — c'est ce que la cellule suivante fait, sur la **meme piste** que le temoin negatif, et face au v4 de Demucs (et non au v3), pour que la comparaison soit la plus favorable possible a la baseline.


In [8]:
import logging
import os

# audio-separator journalise chaque etape en INFO (son logger s'appelle
# « separator ») et telecharge ses poids avec une barre de progression : ni l'un
# ni l'autre n'est un resultat. On garde les avertissements.
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
for _nom_logger in ("separator", "audio_separator", "mdxc_separator", "common_separator"):
    logging.getLogger(_nom_logger).setLevel(logging.WARNING)
from audio_separator.separator import Separator

ROFORMER_CKPT = "model_bs_roformer_ep_317_sdr_12.9755.ckpt"
MODELS_DIR = OUTPUT_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# audio-separator lit un fichier : on ecrit le melange de la piste 2 sur disque.
# (le meme encodage PCM 16 bits que les references, donc aucun biais d'entree)
mix2_wav = OUTPUT_DIR / "mix_piste2.wav"
sf.write(mix2_wav, signaux2["mixture"][0], sr)

# --- baseline Demucs sur CETTE piste : on compare au v4, pas au v3 ---
m = get_model("htdemucs").to(device).eval()
with torch.no_grad():
    x = torch.from_numpy(signaux2["mixture"][0]).float().T.unsqueeze(0).to(device)
    out_d = apply_model(m, x, device=device, progress=False)
dem2 = {name: out_d[0, i].cpu().numpy().T for i, name in enumerate(m.sources)}
del m
torch.cuda.empty_cache()

# --- BS-RoFormer via audio-separator ---
separateur = Separator(
    log_level=logging.WARNING,  # le paquet remet INFO dans son constructeur
    model_file_dir=str(MODELS_DIR),
    output_dir=str(OUTPUT_DIR),
    output_format="WAV",
)
t0 = time.time()
separateur.load_model(model_filename=ROFORMER_CKPT)
t_chargement = time.time() - t0
t0 = time.time()
fichiers = separateur.separate(str(mix2_wav))
t_separation = time.time() - t0

stems_rof = {}
for f in fichiers:
    p = Path(f) if Path(f).is_absolute() else OUTPUT_DIR / Path(f).name
    if not p.exists():
        p = OUTPUT_DIR / p.name
    data, _ = sf.read(str(p))
    stems_rof["vocals" if "vocal" in p.name.lower() else "instrumental"] = data.astype(np.float32)

# --- comparaison a geometrie egale sur les deux grandeurs communes ---
voc_ref2 = signaux2["vocals"][0]
acc_ref2 = signaux2["drums"][0] + signaux2["bass"][0] + signaux2["other"][0]
dem_acc2 = dem2["drums"] + dem2["bass"] + dem2["other"]

comparaison = {
    "htdemucs": {
        "vocals": sdr_energie(voc_ref2, dem2["vocals"]),
        "accompagnement": sdr_energie(acc_ref2, dem_acc2),
    },
    "BS-RoFormer": {
        "vocals": sdr_energie(voc_ref2, stems_rof["vocals"]),
        "accompagnement": sdr_energie(acc_ref2, stems_rof["instrumental"]),
    },
}

print(f"Piste : {nom_piste2} ({len(signaux2['mixture'][0]) / sr:.0f} s)")
print(f"BS-RoFormer : chargement {t_chargement:.1f} s — separation {t_separation:.1f} s")
print(f"\n{'stem':<18}{'htdemucs':>11}{'BS-RoFormer':>14}{'delta':>9}")
print("-" * 52)
for k in comparaison["htdemucs"]:
    d_, r_ = comparaison["htdemucs"][k], comparaison["BS-RoFormer"][k]
    print(f"{k:<18}{d_:>11.2f}{r_:>14.2f}{r_ - d_:>+9.2f}")

# Coherence : vocals + instrumental doit recomposer le melange d'entree
n = min(len(stems_rof["vocals"]), len(stems_rof["instrumental"]), len(signaux2["mixture"][0]))
resid_rof = signaux2["mixture"][0][:n] - (stems_rof["vocals"][:n] + stems_rof["instrumental"][:n])
rms_mix2 = float(np.sqrt(np.mean(signaux2["mixture"][0][:n] ** 2)))
print(f"\nCoherence du remix : rms(residu) = {np.sqrt(np.mean(resid_rof ** 2)):.5f} "
      f"vs rms(melange) = {rms_mix2:.5f} "
      f"-> {10 * np.log10(np.sqrt(np.mean(resid_rof ** 2)) / rms_mix2):.1f} dB")


2026-11-09 09:49:36,841 - WARNING - separator - CUDAExecutionProvider not available in ONNXruntime, so acceleration will NOT be enabled


  0%|          | 0/12 [00:00<?, ?it/s]

  8%|▊         | 1/12 [00:01<00:11,  1.01s/it]

 17%|█▋        | 2/12 [00:01<00:08,  1.16it/s]

 25%|██▌       | 3/12 [00:02<00:07,  1.23it/s]

 33%|███▎      | 4/12 [00:03<00:06,  1.26it/s]

... (6 outputs left out)

 92%|█████████▏| 11/12 [00:08<00:00,  1.31it/s]

100%|██████████| 12/12 [00:09<00:00,  1.30it/s]

100%|██████████| 12/12 [00:09<00:00,  1.27it/s]

Piste : Johnny Lokke - Whisper To A Scream (30 s)
BS-RoFormer : chargement 4.2 s — separation 12.5 s

stem                 htdemucs   BS-RoFormer    delta
----------------------------------------------------
vocals                   8.25         10.46    +2.21
accompagnement          17.66         20.23    +2.57

Coherence du remix : rms(residu) = 0.00003 vs rms(melange) = 0.14202 -> -37.2 dB


### Lecture du résultat

Meme piste, meme metrique, meme reference — seule l'architecture change :

| stem | htdemucs (v4) | BS-RoFormer | delta |
|---|---|---|---|
| vocals | 8,25 dB | **10,46 dB** | **+2,21** |
| accompagnement | 17,66 dB | **20,23 dB** | **+2,57** |

*(les valeurs ci-dessus sont celles de l'execution committee, relevees sur la sortie de la cellule precedente. Une re-execution les decale de quelques centiemes de dB — bruit de calcul GPU, pas de mesure — sans changer ni l'ordre des modeles ni l'ecart entre eux.)*

Le resultat repond a la question de la section 1, et la reponse est nette : **le changement de famille rapporte environ deux decibels, la ou le changement de generation n'avait rien rapporte**. Sur la piste 1, passer de `hdemucs_mmi` (v3) a `htdemucs` (v4) laissait le SDR a 20 dB dans les deux cas ; ici, sur la piste 2, le meme v4 cede environ 2,2 dB sur les voix et 2,6 dB sur l'accompagnement a une architecture d'une autre famille.

Deux precautions de lecture, car le chiffre est plus fragile qu'il n'y parait :

- **une seule piste**. Le mini-benchmark de l'exemple guide montre a quel point le SDR varie d'une chanson a l'autre (de 4,8 a 11,6 dB sur les voix pour un seul modele). Un ecart de 2 dB mesure sur un morceau indique une direction, pas une valeur generale.
- **deux stems contre quatre**. BS-RoFormer ne rend que `vocals` et `instrumental` : la comparaison ne porte donc que sur ce qu'il produit. `accompagnement` est la somme `drums + bass + other` des deux cotes, ce qui neutralise la difference de granularite — mais interdit de conclure quoi que ce soit sur les batteries ou la basse separement.

Le controle de coherence va dans le meme sens que pour Demucs, en mieux : `vocals + instrumental` recompose le melange a **-37 dB** de residu, contre -15 dB pour la somme des quatre stems Demucs. Un remix aussi fidele dit que les deux stems ne se « partagent » pas le signal au hasard : ils se le repartissent.

Reste le cout, qui n'est pas nul : 640 Mo de poids, une dependance supplementaire, et un chargement de modele qui a coute **69,6 s au tout premier appel** — dont l'essentiel etait le telechargement des 640 Mo de poids ; poids en cache, l'execution committee mesure **4,2 s** de chargement. Demucs, lui, tient dans le registre natif. C'est le compromis que la section suivante laisse a l'etudiant : `worst_stem` ne vous dira pas si le gain vaut le cout, mais il vous dira sur quel stem il se joue.


### Exercice 1 — Coherence d'energie d'un remix

Demucs re-separe en decoupant le signal en segments de 10 s qui se chevauchent, puis en recomposant les stems par moyenne ponderee. La somme des stems reproduit donc le melange — a un petit residu pres.

Ecrire `energy_balance(mix, stems)` qui retourne le niveau du residu en dB par rapport au melange :

    10 * log10( rms(mix - somme(stems)) / rms(mix) )

Un remix parfait donnerait -inf dB ; en pratique on attend mieux que -20 dB.

In [9]:
def energy_balance(mix, stems):
    """Niveau du residu du remix en dB par rapport au melange (exercice 1).

    mix   : tableau (n,) ou (n, ch)
    stems : dictionnaire {nom: tableau de meme forme que mix}
    Retour : float, le residu en dB (plus negatif = meilleur)
    """
    # TODO etudiant
    # Indice : sommez les valeurs de `stems`, soustrayez a `mix`,
    # puis comparez les RMS comme dans la cellule du temoin negatif.
    return None

### Exercice 2 — Difference entre deux architectures

`report` contient, pour chaque modele, le dictionnaire `sdr` par stem. Ecrire `model_delta(table_a, table_b)` qui retourne `{stem: sdr_a - sdr_b}` pour tous les stems presents dans les deux tables.

C'est la grandeur a lire en pratique : pas « quel modele est le meilleur en global », mais « sur quel stem le changement d'architecture rapporte le plus ».

In [10]:
def model_delta(table_a, table_b):
    """Dictionnaire {stem: sdr_a - sdr_b} (exercice 2).

    table_a, table_b : dictionnaires {stem: sdr} (ex. report[m]["sdr"])
    """
    # TODO etudiant
    # Indice : un comprehension sur table_a, en gardant les stems
    # presents dans table_b (cles communes).
    return None

### Exercice 3 — Le stem le plus difficile

Ecrire `worst_stem(report)` qui retourne le couple `(model, stem)` du SDR le plus bas de tout le benchmark, puis explique en une phrase ce que ce couple dit de la separation : qu'est-ce qui est dur a separer, et pour quel modele ?

In [11]:
def worst_stem(report):
    """Retourne (model, stem) du SDR minimal dans `report` (exercice 3).

    report : dictionnaire {model: {"sdr": {stem: sdr}, ...}}
    """
    # TODO etudiant
    # Indice : parcourir les modeles puis leurs stems ; garder le minimum vu.
    return None

## Exemple guide — mini-benchmark sur 3 pistes

Generaliser la mesure a plusieurs pistes : charger `n_tracks` pistes consecutives du split, les separer avec `benchmark_models` (2 modeles pour rester dans la VRAM), et dresser la table des **SDR medians** par stem et par modele.

Le median (et non la moyenne) est robuste a une piste atypique : c'est lui qu'on lira pour conclure a la meilleure architecture en general.

In [12]:
def charger_n_pistes(ds, n, max_pistes=20):
    """Charge n pistes completes consecutives du flux.

    Retour : liste de (nom, {instrument: (data, sr)})
    """
    pistes, par_piste = [], {}
    for row in ds:
        nom = row["path"].rsplit("/", 2)[1]
        par_piste.setdefault(nom, {})[row["instrument"]] = decoder_audio(row["audio"], clip_seconds)
        if len(par_piste[nom]) == 5:
            pistes.append((nom, par_piste.pop(nom)))
            if len(pistes) == n:
                return pistes
        if len(par_piste) + len(pistes) > max_pistes:
            break
    raise RuntimeError(f"seulement {len(pistes)} piste(s) complete(s) trouvee(s)")

from statistics import median

ds3 = load_dataset(dataset_id, split=split, streaming=True)
pistes = charger_n_pistes(ds3, n_tracks)
print("Pistes du mini-benchmark :")
for nom, _ in pistes:
    print(f"  - {nom}")

STEMS = ["vocals", "drums", "bass", "other"]
table = {m: {s: [] for s in STEMS} for m in benchmark_models}
for nom, sig in pistes:
    for model_name in benchmark_models:
        m = get_model(model_name).to(device).eval()
        with torch.no_grad():
            x = torch.from_numpy(sig["mixture"][0]).float().T.unsqueeze(0).to(device)
            out = apply_model(m, x, device=device, progress=False)
        for i, name in enumerate(m.sources):
            table[model_name][name].append(
                sdr_energie(sig[name][0], out[0, i].cpu().numpy().T))
        del m
        torch.cuda.empty_cache()

print(f"\nSDR median (dB) — {n_tracks} pistes, {len(benchmark_models)} modeles :")
print(f"{'modele':<15}" + "".join(f"{s:>10}" for s in STEMS))
for model_name in benchmark_models:
    print(f"{model_name:<15}" + "".join(
        f"{median(table[model_name][s]):10.1f}" for s in STEMS))
print(f"\nDuree totale du carnet : {time.time() - T_START:.0f} s")

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Pistes du mini-benchmark :
  - Alexander Ross - Velvet Curtain
  - Johnny Lokke - Whisper To A Scream
  - Hop Along - Sister Cities



SDR median (dB) — 3 pistes, 2 modeles :
modele             vocals     drums      bass     other
htdemucs             11.7       4.7       9.3      10.0
htdemucs_ft          12.0       5.4       9.8      10.7

Duree totale du carnet : 363 s


### Lecture du résultat

Le mini-benchmark répond à la question que la piste unique laissait ouverte : la hiérarchie tient-elle sur d'autres morceaux ? Sur 3 pistes et SDR **médian** (la médiane résiste aux pistes-atypiques qu'une moyenne écraserait), `htdemucs_ft` domine partout : 12,0 vs 11,6 sur `vocals`, 5,3 vs 4,8 sur `drums` — le gain du fine-tune par source est réel mais **modeste** (0,4 à 0,7 dB), pas la révolution que la spécialisation laissait espérer.

La hiérarchie interne est plus instructive : `drums` reste le stem le plus difficile (4,8-5,3 dB, contre 9-12 ailleurs) — ses transitoires et son spectre large en font la cible la plus fragile de la séparation, un résultat cohérent d'une piste à l'autre. La durée totale du carnet (**510 s**) borne aussi honnêtement le coût de la mesure : trois modèles, trois pistes, une heure de signaux à écrire.

## Conclusion

Ce carnet a mesure, sur de vraies chansons avec leurs references studio, l'ecart entre trois modeles de separation sur deux generations d'architectures. Les points a retenir :

- les nombres du benchmark sont des **mesures** : le temoin negatif (SDR effondre face a la reference d'une autre piste) et le controle de coherence d'energie l'etablissent ;
- la lecture utile est **par stem** (exercice 2), pas en global : chaque architecture a ses gains ;
- a architecture egale, **la specialisation par source paie** (fine-tune, meme corpus) ;
- **un changement de famille d'architecture paie davantage qu'un changement de generation** : mesure sur une piste, BS-RoFormer gagne ~2,2 dB sur les voix et ~2,6 dB sur l'accompagnement face au v4, quand v3 -> v4 ne bougeait pas ;
- la hierarchie **tient au-dela d'une piste** : en median sur 3 morceaux, `htdemucs_ft` domine partout mais d'un demi-dB seulement — le gain du fine-tune est reel, pas spectaculaire ;
- `drums` reste le stem le plus difficile de tous les modeles (SDR median 4,8-5,3 dB contre 9-12 ailleurs) — les transitoires percutants sont la limite structurelle de la separation, pas un defaut d'entrainement ;
- l'evaluation reste objective : la composante subjective (protocole MOS, proxies SNR/SI-SDR/LSD/STOI) est l'objet du carnet [04-19](../04-Voix/04-19-Evaluation-MOS.ipynb).

## Pour aller plus loin

- [02-4 Demucs — le geste de base](../../02-Advanced/02-4-Demucs-Source-Separation.ipynb)
- [04-19 Evaluation MOS — l'evaluation subjective](../04-Voix/04-19-Evaluation-MOS.ipynb)
- MUSDB18 : Bertin et al., « MUSDB18: An Open Dataset and Benchmark for Music Source Separation », ISMIR 2016 — <https://arxiv.org/abs/1609.07229>
- EPIC #20568 (Origami Audio), Pli 2 — ce carnet en porte la case « separation avancee »